# Building a Crew to Prepare for Meetings

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

### Install Dependencies

Uncomment the following lines to install the required packages.

In [2]:
# Create reusable loading animation class
import os
import sys
import time
import threading

class LoadingAnimation:
    def __init__(self):
        self.stop_event = threading.Event()
        self.animation_thread = None

    def _animate(self, message="Loading"):
        chars = "/—\\|"
        while not self.stop_event.is_set():
            for char in chars:
                sys.stdout.write('\r' + message + '... ' + char)
                sys.stdout.flush()
                time.sleep(0.1)
                if self.stop_event.is_set():
                    sys.stdout.write("\n")
                    break

    def start(self, message="Loading"):
        self.stop_event.clear()
        self.animation_thread = threading.Thread(target=self._animate, args=(message,))
        self.animation_thread.daemon = True
        self.animation_thread.start()

    def stop(self, completion_message="Complete"):
        self.stop_event.set()
        if self.animation_thread:
            self.animation_thread.join()
        print(f"\r{completion_message} ✓")

# Use the animation for pip install
loader = LoadingAnimation()
loader.start("Installing")
%pip install -r requirements.txt -q
loader.stop("Installation complete")

Installing... |Note: you may need to restart the kernel to use updated packages.

Installation complete ✓


### Helper Functions

In [3]:
import dotenv
from dotenv import dotenv_values

# Define a fake `load_dotenv` function
def _load_dotenv(*args, **kwargs):
    env_path = kwargs.get('dotenv_path', '.env')  # Default to '.env'
    parsed_env = dotenv_values(env_path)

    # Manually set valid key-value pairs
    for key, value in parsed_env.items():
        if key and value:  # Check for valid key-value pairs
            os.environ[key] = value

# Replace the original load_dotenv function
dotenv.load_dotenv = _load_dotenv

# Load environment variables
dotenv.load_dotenv()

### Initialization and Setup
Initial imports for the CrewAI Flow and Crew and setting up the environment

In [5]:
# Importing necessary libraries
import os
import yaml

# Importing Crew related components
from crewai import Agent, Task, Crew, Process
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

# Apply a patch to allow nested asyncio loops in Jupyter
import nest_asyncio
nest_asyncio.apply()

In [6]:
from typing import Type
from pydantic import BaseModel, Field
from openai import OpenAI
from pinecone import Pinecone

from crewai.tools import BaseTool

class SemanticSearchInput(BaseModel):
    """Input schema for SemanticSearchTool."""
    query: str = Field(..., description="The text query to semantically search for.")

class SemanticSearchTool(BaseTool):
    name: str = "Semantic Search Tool"
    description: str = (
        "A tool for performing semantic search using Pinecone vector database. "
        "It converts text queries into embeddings and finds semantically similar content."
    )
    args_schema: Type[BaseModel] = SemanticSearchInput

    def _run(self, query: str) -> str:
        client = OpenAI()
        embedding = client.embeddings.create(input=query, model="text-embedding-3-large")

        pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))
        index = pc.Index(host=os.getenv('PINECONE_INDEX_HOST'))

        result = index.query(
            namespace="transcripts",
            vector=embedding.data[0].embedding,
            top_k=3,
            include_metadata=True
        )

        search_results = [item['metadata']['text'] for item in result['matches']]
        return "\n\n".join(search_results)


In [7]:
# Create output directory if it doesn't exist
os.makedirs('output', exist_ok=True)

# Load agent and task configurations from YAML files
with open('config/agents.yml', 'r') as f:
    agents_config = yaml.safe_load(f)

with open('config/tasks.yml', 'r') as f:
    tasks_config = yaml.safe_load(f)

# Define the agents for our meeting preparation crew
researcher = Agent(
    config=agents_config['researcher'],
    verbose=True,
    tools=[SerperDevTool(), ScrapeWebsiteTool(), SemanticSearchTool()]  # Web search and document analysis tools
)

analyst = Agent(
    config=agents_config['analyst'],
    max_iter=3,
    verbose=True,
)

# Define the tasks for our crew
historical_context_task = Task(
    config=tasks_config['historical_context_task'],
    agent=researcher,
)

research_task = Task(
    config=tasks_config['research_task'],
    agent=researcher,
)

preparation_task = Task(
    config=tasks_config['preparation_task'],
    agent=analyst,
    output_file="output/meeting_brief.md"
)

# Create the crew
meeting_prep_crew = Crew(
    agents=[researcher, analyst],
    tasks=[historical_context_task, research_task, preparation_task],
    process=Process.sequential,
    memory=False,
    verbose=True
)

In [8]:
# If inputs are not provided, ask for them
meeting_topic = input("Enter the meeting topic: ")
participants = input("Enter the meeting participants: ")

print(f"\nPreparing for meeting about: {meeting_topic}")
print(f"Participants: {participants}\n")

# Run the crew with the provided inputs
result = meeting_prep_crew.kickoff(inputs={
    "topic": meeting_topic,
    "participants": participants
})

print("\n\nMeeting brief has been saved to output/meeting_brief.md")
# Display the markdown content in a formatted way
from IPython.display import Markdown, display

# Display the raw result as formatted markdown
display(Markdown(result.raw))


Preparing for meeting about: Initial sales call with google, to talk about CrewAI enterprise and its observabiltiy features
Participants: Meeting with the CTO and CIO of google to talk about security convernso nrunning agents and what features we offer

 
[2025-04-14 10:03:56][🚀 CREW 'CREW' STARTED, 4E499E99-0124-4A70-8A73-757F50F39E78]: 2025-04-14 10:03:56.566049
 
[2025-04-14 10:03:56][📋 TASK STARTED: SEARCH AND ANALYZE PREVIOUS MEETING RECORDS AND INTERACTIONS RELATED TO INITIAL SALES CALL WITH GOOGLE, TO TALK ABOUT CREWAI ENTERPRISE AND ITS OBSERVABILTIY FEATURES AND MEETING WITH THE CTO AND CIO OF GOOGLE TO TALK ABOUT SECURITY CONVERNSO NRUNNING AGENTS AND WHAT FEATURES WE OFFER.
YOUR OBJECTIVE IS TO EXTRACT VALUABLE HISTORICAL CONTEXT THAT CAN INFORM OUR CURRENT MEETING STRATEGY:
1. PREVIOUS INTERACTIONS:
   - KEY DISCUSSION POINTS FROM PAST MEETINGS
   - COMMITMENTS MADE AND THEIR FULFILLMENT STATUS
   - EVOLUTION OF POSITIONS AND RELATIONSHIPS OVER TIME

2. PATTERN RECOGNITION

# Strategic Meeting Brief: Initial Sales Call with Google CTO and CIO

## ONE-PAGER OVERVIEW

**Meeting Objective:**  
Engage Google executives to discuss CrewAI’s enterprise observability features, focusing on security and integration capabilities, ultimately aiming for a collaborative pilot project.

**Success Criteria:**  
- Positive feedback on security protocols and integration capabilities.
- Agreement on next steps towards a proof-of-concept (POC).
- Clear alignment on follow-up commitments.

**Critical Background:**  
- CrewAI has advanced observability features targeted at enterprise needs.
- Concerns about security and integration have been raised in previous interactions.
- Google prioritizes ethical AI deployment and robust security measures.

**Top 3 Recommended Strategies:**
1. **Emphasize Security:** Use case studies showcasing successful secure deployments.
2. **Cultivate Relationships:** Leverage existing rapport to foster trust and openness.
3. **Data-Driven Approach:** Provide concrete metrics and outcomes from existing implementations.

**Key Risks and Mitigation Tactics:**
- **Risk:** Potential doubts about integration complexity.  
  **Mitigation:** Offer testimonials and case studies demonstrating ease of integration.
- **Risk:** Resistance due to security concerns.  
  **Mitigation:** Present detailed security protocols and compliance standards.

---

## PARTICIPANT STRATEGY

### Prabhakar Raghavan (CTO of Google)
- **Key Interests:** Ethical AI deployment, security measures.
- **Tailored Messaging:** "CrewAI’s observability features align with your focus on ethical performance, ensuring compliance while maintaining operational efficacy."
- **Points of Leverage:** Discuss CrewAI’s robust privacy controls and its ability to monitor agent performance seamlessly.
- **Relationship-Building:** Highlight past positive interactions and express willingness to adapt offerings to fit Google’s ethical guidelines.

### Will Grannis (CIO of Google Cloud)
- **Key Interests:** Practical integration solutions, security of AI technologies.
- **Tailored Messaging:** "Our observability tools enhance operational oversight, ensuring AI deployment is both secure and user-friendly."
- **Points of Leverage:** Emphasize successful integrations with existing cloud infrastructures.
- **Relationship-Building:** Engage in dialogue about how mutual goals align in customer-centric approaches.

---

## TALKING POINTS AND RESPONSES

### Prioritized Talking Points:
1. **Security and Protocol Compliance:** "CrewAI implements strict privacy protocols, ensuring data security and compliance with regulations."
2. **Integration Flexibility:** "Our solutions are designed for seamless integration with existing Google Cloud infrastructures."
3. **Customer Success Stories:** "Case studies highlighting successful deployments with measurable outcomes."

### Anticipated Questions with Responses:
- **Question:** "How secure is CrewAI’s technology regarding data handling?"  
  **Response:** "CrewAI employs state-of-the-art security measures, validated by client success stories and compliance certifications."
  
- **Question:** "What about scalability for larger datasets?"  
  **Response:** "Our platform is built for scalability, with case studies demonstrating handling of large datasets effectively without latency."

### Objection Handling Framework:
- **Objection:** "Integration seems complex."  
  **Framework:** "We have a dedicated support team to assist with integration, plus documentation and case studies underline our historically smooth deployments."

### Strategic Silence Recommendations:
- Use pauses effectively after presenting key data points to allow time for absorption and questions.

---

## EXECUTION PLAN

### Recommended Meeting Flow:
1. **Introduction and Objectives (5 mins):** Outline meeting goals and establish rapport.
2. **Presentation of Key Features (15 mins):** Focus on security, integration, and observability features.
3. **Open Floor for Questions (10 mins):** Engage directly with each participant’s queries and concerns.
4. **Discussion of Next Steps (10 mins):** Propose POC commitments and outline follow-up actions.

### Critical Decision Points:
- Assess interest levels in CrewAI’s offerings based on feedback.
- Confirm commitment to exploring a POC to address security concerns.

### Proposed Next Steps and Commitments:
- Follow up with detailed documentation on case studies and integration processes.
- Schedule a technical follow-up meeting within two weeks.

### Follow-Up Strategy and Timeline:
- **Immediate Follow-Up:** Send thank-you notes with summaries of discussed points within 24 hours.
- **Documentation:** Deliver case studies within one week.
- **Check-In Call:** Schedule within two weeks to discuss feedback and next steps.

This brief provides a structured path to navigate the conversation effectively while positioning CrewAI to meet Google’s comprehensive security and observability needs.